# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring the FAIR^2 dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")
print(f"RecordSet IDs: {[r.id for r in metadata.record_sets]}")

## 2. Data Overview
Review available record sets, fields, and their IDs by iterating over the dataset structure as defined in the Croissant schema.

This helps in understanding the main tables and available fields for extraction and analysis.

In [ ]:
print("Available Record Sets:")
for rs in metadata.record_sets:
    print(f"- RecordSet ID: {rs.id}, Name: {rs.name}")
    print("  Fields:")
    for f in rs.fields:
        print(f"     • Field: {f.name:35} | id: {f.id} | type: {f.data_type}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.

Below we demonstrate generic extraction for all main record sets.

In [ ]:
# Use RecordSet @ids as required by Croissant
record_set_ids = [rs.id for rs in metadata.record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        dataframes[record_set_id] = pd.DataFrame(records)
        print(f"Loaded {len(records)} records from RecordSet {record_set_id}")
    except Exception as e:
        print(f"Could not load RecordSet {record_set_id}: {e}")

if dataframes:
    # Pick the first non-empty frame for demo
    first_rs_id = next(rsid for rsid, df in dataframes.items() if not df.empty)
    print(f"\nColumns in {first_rs_id}: {dataframes[first_rs_id].columns.tolist()}")
    display(dataframes[first_rs_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

For demonstration, we select a numeric field (for example, 'cr:field:MW_kDa' – molecular weight in kDa) and a group field (for example, 'cr:field:sample_source' if it exists) from the main protein record set.

In [ ]:
# Choose a record set and field IDs for the following EDA steps:
# You may need to adjust these field IDs according to the listing in Section 2

# Example: Suppose 'cr:recordSet:proteins' is the main set, 'cr:field:MW_kDa' is molecular weight, 'cr:field:sample_source' is a grouping field

main_rs_id = first_rs_id # Use the first loaded record set from above
df = dataframes[main_rs_id]

# Find the first numeric field in the record set
numeric_field_id = None
for f in [f for f in metadata.record_sets if f.id == main_rs_id][0].fields:
    if f.data_type.lower() in {"float", "integer", "number"}:
        numeric_field_id = f.id
        break
print(f"Selected numeric field for EDA: {numeric_field_id}")
if numeric_field_id is None:
    raise ValueError("No numeric field found in the selected record set.")

# Ensure numeric type
df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')

# Remove rows with missing values in the numeric field
filtered_df = df[df[numeric_field_id].notnull()]
# Set a threshold at the numeric field's 90th percentile for demonstration
threshold = filtered_df[numeric_field_id].quantile(0.90)
filtered_df = filtered_df[filtered_df[numeric_field_id] > threshold]
print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
display(filtered_df.head())

# Normalize the numeric field
filtered_df[f"{numeric_field_id}_normalized"] = (
    filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
) / filtered_df[numeric_field_id].std()

print(f"Normalized {numeric_field_id} for filtered records:")
display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Try grouping by a categorical field, e.g., the first non-numeric field
group_field_id = None
for f in [f for f in metadata.record_sets if f.id == main_rs_id][0].fields:
    if f.data_type.lower() not in {"float", "integer", "number"}:
        group_field_id = f.id
        break

if group_field_id and group_field_id in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
    print(f"Grouped data by {group_field_id} (showing top 5 groups):")
    display(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
# Histogram of the numeric field
plt.figure(figsize=(8,4))
df[numeric_field_id].hist(bins=30, grid=False, alpha=0.7)
plt.title(f"Distribution of {numeric_field_id}")
plt.xlabel(numeric_field_id)
plt.ylabel("Frequency")
plt.show()

# If there is a group field, make a boxplot
if group_field_id and group_field_id in df.columns:
    plt.figure(figsize=(10,5))
    # Take only top 8 categories for clarity (if any)
    top_groups = df[group_field_id].value_counts().head(8).index
    boxplot_data = [df[df[group_field_id]==val][numeric_field_id].dropna() for val in top_groups]
    plt.boxplot(boxplot_data, labels=top_groups)
    plt.title(f"{numeric_field_id} by {group_field_id} (Top 8)")
    plt.xlabel(group_field_id)
    plt.ylabel(numeric_field_id)
    plt.show()

## 6. Conclusion
In this notebook, we demonstrated how to explore the FAIR^2 dataset using the Croissant schema and the `mlcroissant` library. We loaded and reviewed the metadata, examined available record sets and fields using their `@id`s, extracted a data table for analysis, performed EDA including filtering and normalization, and visualized key numeric fields. These steps provide a foundation for deeper analysis or integration of the dataset into downstream data science workflows.